In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def rossler4_rhs(t, s, a=0.25, b=3.0, c=0.5, d=0.05):
    x, y, z, w = s
    dx = -(y + z)
    dy = x + a*y + w
    dz = b + x*z
    dw = -c*z + d*w
    return [dx, dy, dz, dw]

# Integreren
a, b, c, d = 0.25, 3.0, 0.5, 0.05
s0 = [1.0, 0.0, 0.0, 0.0]
t_span = (0.0, 400.0)
t_eval = np.linspace(*t_span, 200_000)

sol = solve_ivp(rossler4_rhs, t_span, s0, t_eval=t_eval, args=(a,b,c,d), rtol=1e-9, atol=1e-12)
x, y, z, w = sol.y

# Transient weggooien
skip = int(0.2 * x.size)
x, y, z, w = x[skip:], y[skip:], z[skip:], w[skip:]

# 3D + kleur = w
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
p = ax.scatter(x[::10], y[::10], z[::10], c=w[::10], s=0.2)  # downsample voor snelheid
fig.colorbar(p, ax=ax, shrink=0.6, label="w")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.set_title("4D Rössler: (x,y,z) met w als kleur")
plt.show()


In [ ]:
pairs = [("x","y", x,y), ("x","z", x,z), ("x","w", x,w),
         ("y","z", y,z), ("y","w", y,w), ("z","w", z,w)]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (n1, n2, u, v) in zip(axes.ravel(), pairs):
    ax.plot(u[::10], v[::10], lw=0.2)
    ax.set_xlabel(n1); ax.set_ylabel(n2)
    ax.grid(True, alpha=0.3)
fig.suptitle("4D Rössler: 2D projecties (downsampled)")
plt.tight_layout()
plt.show()


In [ ]:
# Poincaré: w = 0 crossing met dw/dt > 0
w_series = w
dw_series = np.gradient(w_series, sol.t[skip:])  # ruwe dw/dt

cross = np.where((w_series[:-1] < 0) & (w_series[1:] >= 0) & (dw_series[1:] > 0))[0]
# lineaire interpolatie naar w=0
alpha = -w_series[cross] / (w_series[cross+1] - w_series[cross])
xp = x[cross] + alpha*(x[cross+1]-x[cross])
yp = y[cross] + alpha*(y[cross+1]-y[cross])
zp = z[cross] + alpha*(z[cross+1]-z[cross])

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(xp, yp, zp, s=2, alpha=0.6)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.set_title("Poincaré sectie: w=0, dw/dt>0")
plt.show()


In [ ]:
X = np.vstack([x, y, z, w]).T
X = X - X.mean(axis=0, keepdims=True)

# PCA via SVD
U, S, Vt = np.linalg.svd(X, full_matrices=False)
PC = X @ Vt.T  # kolommen = principal components

plt.figure(figsize=(6,6))
plt.plot(PC[::10,0], PC[::10,1], lw=0.2)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.grid(True, alpha=0.3)
plt.title("PCA projectie van 4D attractor naar 2D")
plt.show()
